# Code Generation from Python to R using GPT, Claude and CodeQwen (HuggingFace Endpoint)

The requirement: use an Open Source model to generate high performance R code from Python code

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important - Pause Endpoints when not in use</h1>
            <span style="color:#900;">
            When using HuggingFace endpoints for this project, always be sure to stop or pause the endpoints when you are done to avoid accruing unnecessary running cost. The costs are very low as long as you only run the endpoint when you're using it. Navigate to the HuggingFace endpoint UI <a href="https://ui.endpoints.huggingface.co/">here,</a> open your endpoint, and click Pause to put it on pause so you no longer pay for it.  
<br/><br/>
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display, update_display
import gradio as gr
import subprocess

In [2]:
# environment

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

In [3]:
# initialize

openai = OpenAI()
claude = anthropic.Anthropic()
OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"

In [4]:
system_message = "You are an assistant that reimplements Python code in high performance R for an M1 Mac. "
system_message += "Respond only with R code; use comments sparingly and do not provide any explanation other than occasional comments. "
system_message += "The R response needs to produce an identical output in the fastest possible time. Keep implementations of random number generators identical so that results match exactly."

In [5]:
def user_prompt_for(python):
    user_prompt = "Rewrite this Python code in R with the fastest possible implementation that produces identical output in the least time. "
    user_prompt += "Respond only with R code; do not explain your work other than a few comments. "
    user_prompt += "Pay attention to number types to ensure no int overflows. Remember to include all necessary R packages.\n\n"
    user_prompt += python
    return user_prompt

In [6]:
def messages_for(python):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [7]:
# write to a file called optimized.cpp

def write_output(R):
    code = R.replace("```r","").replace("```","")
    with open("optimized.R", "w") as f:
        f.write(code)

In [8]:
def optimize_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    write_output(reply)

In [9]:
def optimize_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)

In [10]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [11]:
exec(pi)

Result: 3.141592658589
Execution Time: 7.904888 seconds


In [12]:
optimize_gpt(pi)

```r
library(parallel)

# Define the calculation function
calculate <- function(iterations, param1, param2) {
  result <- 1.0
  # Use vectorized operations for performance
  i <- 1:iterations
  result <- result - sum(1 / (i * param1 - param2)) + sum(1 / (i * param1 + param2))
  return(result)
}

# Measure execution time
start_time <- proc.time()
result <- calculate(100000000, 4, 1) * 4
end_time <- proc.time()

cat(sprintf("Result: %.12f\n", result))
cat(sprintf("Execution Time: %.6f seconds\n", (end_time - start_time)[3]))
```

In [13]:
exec(pi)

Result: 3.141592658589
Execution Time: 7.806247 seconds


In [14]:
!Rscript optimized.R

Result: 3.141592658587
Execution Time: 1.002000 seconds


In [15]:
optimize_claude(pi)

```r
library(microbenchmark)

calculate <- function(iterations, param1, param2) {
  i <- 1:iterations
  j1 <- i * param1 - param2
  j2 <- i * param1 + param2
  sum(1/j2 - 1/j1)
}

start_time <- Sys.time()
result <- calculate(100000000, 4, 1) * 4
end_time <- Sys.time()

cat(sprintf("Result: %.12f\n", result))
cat(sprintf("Execution Time: %.6f seconds\n", as.numeric(end_time - start_time, units="secs")))
```

In [16]:
!Rscript optimized.R

Result: -0.858407341903
Execution Time: 1.102452 seconds


In [17]:
python_hard = """# Be careful to support large number sizes

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [18]:
exec(python_hard)

Total Maximum Subarray Sum (20 runs): 10980
Execution Time: 24.156683 seconds


In [19]:
optimize_gpt(python_hard)

```r
# Required library
library(microbenchmark)

# Define the Linear Congruential Generator
lcg <- function(seed, a=1664525, c=1013904223, m=2^32) {
  value <- seed
  gen <- function() {
    value <<- (a * value + c) %% m
    value
  }
  return(gen)
}

# Calculate the maximum subarray sum for a given sequence of random numbers
max_subarray_sum <- function(n, seed, min_val, max_val) {
  lcg_gen <- lcg(seed)
  random_numbers <- numeric(n)
  for (i in seq_len(n)) {
    random_numbers[i] <- (lcg_gen() %% (max_val - min_val + 1)) + min_val
  }
  
  max_sum <- -Inf
  for (i in seq_len(n)) {
    current_sum <- 0
    for (j in i:n) {
      current_sum <- current_sum + random_numbers[j]
      if (current_sum > max_sum) {
        max_sum <- current_sum
      }
    }
  }
  max_sum
}

# Calculate the total of maximum subarray sums over multiple runs
total_max_subarray_sum <- function(n, initial_seed, min_val, max_val) {
  total_sum <- 0
  lcg_gen <- lcg(initial_seed)
  for (run in seq_len(20)) {
 

In [20]:
!Rscript optimized.R

Warning message:
In microbenchmark(result <- total_max_subarray_sum(n, initial_seed,  :
  less accurate nanosecond times to avoid potential integer overflows
Total Maximum Subarray Sum (20 runs): 10980 
Execution Time: 2.802812e-05 seconds


In [21]:
optimize_claude(python_hard)

```r
library(bit64)

lcg <- function(seed, a = 1664525L, c = 1013904223L, m = 2^32) {
  value <- as.integer64(seed)
  a <- as.integer64(a)
  c <- as.integer64(c)
  m <- as.integer64(m)
  function() {
    value <<- (a * value + c) %% m
    value
  }
}

max_subarray_sum <- function(n, seed, min_val, max_val) {
  lcg_gen <- lcg(seed)
  random_numbers <- (sapply(1:n, function(x) lcg_gen()) %% (max_val - min_val + 1L) + min_val)
  max_sum <- -Inf
  cumsum_array <- cumsum(random_numbers)
  for (i in 1:n) {
    current_sums <- cumsum_array[i:n] - c(0, cumsum_array[i:(n-1)])
    max_sum <- max(max_sum, max(current_sums))
  }
  max_sum
}

total_max_subarray_sum <- function(n, initial_seed, min_val, max_val) {
  lcg_gen <- lcg(initial_seed)
  seeds <- sapply(1:20, function(x) lcg_gen())
  sum(sapply(seeds, function(seed) max_subarray_sum(n, seed, min_val, max_val)))
}

# Parameters
n <- 10000L        # Number of random numbers
initial_seed <- 42L # Initial seed for the LCG
min_val <- -10L    # M

In [22]:
!Rscript optimized.R

Loading required package: bit

Attaching package: ‘bit’

The following object is masked from ‘package:base’:

    xor

Attaching package bit64
package:bit64 (c) 2011-2017 Jens Oehlschlaegel
creators: integer64 runif64 seq :
coercion: as.integer64 as.vector as.logical as.integer as.double as.character as.bitstring
logical operator: ! & | xor != == < <= >= >
arithmetic operator: + - * / %/% %% ^
math: sign abs sqrt log log2 log10
math: floor ceiling trunc round
querying: is.integer64 is.vector [is.atomic} [length] format print str
values: is.na is.nan is.finite is.infinite
aggregation: any all min max range sum prod
cumulation: diff cummin cummax cumsum cumprod
access: length<- [ [<- [[ [[<-
combine: c rep cbind rbind as.data.frame
WARNING don't use as subscripts
WARNING semantics differ from integer
for more help type ?bit64

Attaching package: ‘bit64’

The following object is masked from ‘package:utils’:

    hashtab

The following objects are masked from ‘package:base’:

    :, %in%, 

In [23]:
def stream_gpt(python):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(python), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace('```r\n','').replace('```','')

In [24]:
def stream_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace('```r\n','').replace('```','')

In [25]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far        

In [26]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=10, value=python_hard)
        R = gr.Textbox(label="R code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
        convert = gr.Button("Convert code")

    convert.click(optimize, inputs=[python, model], outputs=[R])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [27]:
def execute_python(code):
    try:
        output = io.StringIO()
        sys.stdout = output
        exec(code)
    finally:
        sys.stdout = sys.__stdout__
    return output.getvalue()

In [28]:
def execute_R(code):
    write_output(code)
    # compiler_cmd = ["clang++", "-O3", "-std=c++17", "-march=armv8.3-a", "-o", "optimized", "optimized.cpp"]
    try:
        # compile_result = subprocess.run(compiler_cmd, check=True, text=True, capture_output=True)
        run_cmd = ["Rscript","optimized.R"]
        run_result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [29]:
css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
"""

In [30]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to R")
    with gr.Row():
        python = gr.Textbox(label="Python code:", value=python_hard, lines=10)
        R = gr.Textbox(label="R code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
    with gr.Row():
        convert = gr.Button("Convert code")
    with gr.Row():
        python_run = gr.Button("Run Python")
        R_run = gr.Button("Run R")
    with gr.Row():
        python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
        R_out = gr.TextArea(label="R result:", elem_classes=["R"])

    convert.click(optimize, inputs=[python, model], outputs=[R])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    R_run.click(execute_R, inputs=[R], outputs=[R_out])

ui.launch(inbrowser=True,debug=True)

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
<|im_start|>system
You are an assistant that reimplements Python code in high performance R for an M1 Mac. Respond only with R code; use comments sparingly and do not provide any explanation other than occasional comments. The R response needs to produce an identical output in the fastest possible time. Keep implementations of random number generators identical so that results match exactly.<|im_end|>
<|im_start|>user
Rewrite this Python code in R with the fastest possible implementation that produces identical output in the least time. Respond only with R code; do not explain your work other than a few comments. Pay attention to number types to ensure no int overflows. Remember to include all necessary R packages.


import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (

In [43]:
from huggingface_hub import login, InferenceClient
from transformers import AutoTokenizer

In [44]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [45]:
code_qwen = "Qwen/CodeQwen1.5-7B-Chat"
code_gemma = "google/codegemma-7b-it"
CODE_QWEN_URL = "https://hvz8pil4tmzodclp.us-east4.gcp.endpoints.huggingface.cloud"
CODE_GEMMA_URL = "https://c5hggiyqachmgnqg.us-east-1.aws.endpoints.huggingface.cloud"

In [46]:
tokenizer = AutoTokenizer.from_pretrained(code_qwen)
messages = messages_for(pi)
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [47]:
messages

[{'role': 'system',
  'content': 'You are an assistant that reimplements Python code in high performance R for an M1 Mac. Respond only with R code; use comments sparingly and do not provide any explanation other than occasional comments. The R response needs to produce an identical output in the fastest possible time. Keep implementations of random number generators identical so that results match exactly.'},
 {'role': 'user',
  'content': 'Rewrite this Python code in R with the fastest possible implementation that produces identical output in the least time. Respond only with R code; do not explain your work other than a few comments. Pay attention to number types to ensure no int overflows. Remember to include all necessary R packages.\n\n\nimport time\n\ndef calculate(iterations, param1, param2):\n    result = 1.0\n    for i in range(1, iterations+1):\n        j = i * param1 - param2\n        result -= (1/j)\n        j = i * param1 + param2\n        result += (1/j)\n    return resul

In [48]:
text

'<|im_start|>system\nYou are an assistant that reimplements Python code in high performance R for an M1 Mac. Respond only with R code; use comments sparingly and do not provide any explanation other than occasional comments. The R response needs to produce an identical output in the fastest possible time. Keep implementations of random number generators identical so that results match exactly.<|im_end|>\n<|im_start|>user\nRewrite this Python code in R with the fastest possible implementation that produces identical output in the least time. Respond only with R code; do not explain your work other than a few comments. Pay attention to number types to ensure no int overflows. Remember to include all necessary R packages.\n\n\nimport time\n\ndef calculate(iterations, param1, param2):\n    result = 1.0\n    for i in range(1, iterations+1):\n        j = i * param1 - param2\n        result -= (1/j)\n        j = i * param1 + param2\n        result += (1/j)\n    return result\n\nstart_time = t

In [57]:
client = InferenceClient(CODE_QWEN_URL, token=hf_token)
stream = client.text_generation(text, stream=True, details=True, max_new_tokens=3000)
for r in stream:
    print(r.token.text, end = "")

In [58]:
def stream_code_qwen(python):
    tokenizer = AutoTokenizer.from_pretrained(code_qwen)
    messages = messages_for(python)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    client = InferenceClient(CODE_QWEN_URL, token=hf_token)
    stream = client.text_generation(text, stream=True, details=True, max_new_tokens=3000)
    result = ""
    for r in stream:
        result += r.token.text
        yield result    

In [59]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    elif model=="CodeQwen":
        result = stream_code_qwen(python)
    else:
        raise ValueError("Unknown model")
    for stream_so_far in result:
        yield stream_so_far    

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Thank you to @CloudLlama for an amazing contribution</h2>
            <span style="color:#090;">
                A student has contributed a chunk of code to improve this, in the next 2 cells. You can now select which Python porgram to run,
                and a compiler is automatically selected that will work on PC, Windows and Mac. Massive thank you @CloudLlama!
            </span>
        </td>
    </tr>
</table>

In [60]:
def select_sample_program(sample_program):
    if sample_program=="pi":
        return pi
    elif sample_program=="python_hard":
        return python_hard
    else:
        return "Type your Python program here"

In [54]:
import platform

VISUAL_STUDIO_2022_TOOLS = "C:\\Program Files\\Microsoft Visual Studio\\2022\\Community\\Common7\Tools\\VsDevCmd.bat"
VISUAL_STUDIO_2019_TOOLS = "C:\\Program Files (x86)\\Microsoft Visual Studio\\2019\\BuildTools\\Common7\\Tools\\VsDevCmd.bat"

simple_R = """
#include <iostream>

int main() {
    std::cout << "Hello";
    return 0;
}
"""

def run_cmd(command_to_run):
    try:
        run_result = subprocess.run(command_to_run, check=True, text=True, capture_output=True)
        return run_result.stdout if run_result.stdout else "SUCCESS"
    except:
        return ""

def c_compiler_cmd(filename_base):
    my_platform = platform.system()
    my_compiler = []

    try:
        with open("simple.cpp", "w") as f:
            f.write(simple_cpp)
            
        if my_platform == "Windows":
            if os.path.isfile(VISUAL_STUDIO_2022_TOOLS):
                if os.path.isfile("./simple.exe"):
                    os.remove("./simple.exe")
                compile_cmd = ["cmd", "/c", VISUAL_STUDIO_2022_TOOLS, "&", "cl", "simple.cpp"]
                if run_cmd(compile_cmd):
                    if run_cmd(["./simple.exe"]) == "Hello":
                        my_compiler = ["Windows", "Visual Studio 2022", ["cmd", "/c", VISUAL_STUDIO_2022_TOOLS, "&", "cl", f"{filename_base}.cpp"]]
        
            if not my_compiler:
                if os.path.isfile(VISUAL_STUDIO_2019_TOOLS):
                    if os.path.isfile("./simple.exe"):
                        os.remove("./simple.exe")
                    compile_cmd = ["cmd", "/c", VISUAL_STUDIO_2019_TOOLS, "&", "cl", "simple.cpp"]
                    if run_cmd(compile_cmd):
                        if run_cmd(["./simple.exe"]) == "Hello":
                            my_compiler = ["Windows", "Visual Studio 2019", ["cmd", "/c", VISUAL_STUDIO_2019_TOOLS, "&", "cl", f"{filename_base}.cpp"]]
    
            if not my_compiler:
                my_compiler=[my_platform, "Unavailable", []]
                
        elif my_platform == "Linux":
            if os.path.isfile("./simple"):
                os.remove("./simple")
            compile_cmd = ["g++", "simple.cpp", "-o", "simple"]
            if run_cmd(compile_cmd):
                if run_cmd(["./simple"]) == "Hello":
                    my_compiler = ["Linux", "GCC (g++)", ["g++", f"{filename_base}.cpp", "-o", f"{filename_base}" ]]
    
            if not my_compiler:
                if os.path.isfile("./simple"):
                    os.remove("./simple")
                compile_cmd = ["clang++", "simple.cpp", "-o", "simple"]
                if run_cmd(compile_cmd):
                    if run_cmd(["./simple"]) == "Hello":
                        my_compiler = ["Linux", "Clang++", ["clang++", f"{filename_base}.cpp", "-o", f"{filename_base}"]]
        
            if not my_compiler:
                my_compiler=[my_platform, "Unavailable", []]
    
        elif my_platform == "Darwin":
            if os.path.isfile("./simple"):
                os.remove("./simple")
            compile_cmd = ["clang++", "-Ofast", "-std=c++17", "-march=armv8.5-a", "-mtune=apple-m1", "-mcpu=apple-m1", "-o", "simple", "simple.cpp"]
            if run_cmd(compile_cmd):
                if run_cmd(["./simple"]) == "Hello":
                    my_compiler = ["Macintosh", "Clang++", ["clang++", "-Ofast", "-std=c++17", "-march=armv8.5-a", "-mtune=apple-m1", "-mcpu=apple-m1", "-o", f"{filename_base}", f"{filename_base}.cpp"]]
    
            if not my_compiler:
                my_compiler=[my_platform, "Unavailable", []]
    except:
        my_compiler=[my_platform, "Unavailable", []]
        
    if my_compiler:
        return my_compiler
    else:
        return ["Unknown", "Unavailable", []]


In [61]:
# compiler_cmd = c_compiler_cmd("optimized")

with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to R")
    with gr.Row():
        python = gr.Textbox(label="Python code:", value=python_hard, lines=10)
        R = gr.Textbox(label="R code:", lines=10)
    with gr.Row():
        with gr.Column():
            sample_program = gr.Radio(["pi", "python_hard"], label="Sample program", value="python_hard")
            model = gr.Dropdown(["GPT", "Claude", "CodeQwen"], label="Select model", value="GPT")
        # with gr.Column():
        #     architecture = gr.Radio([compiler_cmd[0]], label="Architecture", interactive=False, value=compiler_cmd[0])
        #     compiler = gr.Radio([compiler_cmd[1]], label="Compiler", interactive=False, value=compiler_cmd[1])
    with gr.Row():
        convert = gr.Button("Convert code")
    with gr.Row():
        python_run = gr.Button("Run Python")
        # if not compiler_cmd[1] == "Unavailable":
        R_run = gr.Button("Run R")
        # else:
        #     R_run = gr.Button("No compiler to run R", interactive=False)
    with gr.Row():
        python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
        R_out = gr.TextArea(label="R result:", elem_classes=["R"])

    sample_program.change(select_sample_program, inputs=[sample_program], outputs=[python])
    convert.click(optimize, inputs=[python, model], outputs=[R])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    R_run.click(execute_R, inputs=[R], outputs=[R_out])

ui.launch(inbrowser=True)